In [5]:
# 03 Machine Learning: Student Performance
# Regression target: G3 (0 to 20). Classification target: passed (G3 >= 10).
# Feature set A = all inputs. Feature set B = without G1 and G2.

In [3]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from aislca.data.ingestion import load_config, load_processed_data
from aislca.data import features
from aislca.ml import train, evaluate


config = load_config()
pass_mark = config["target"]["pass_mark"]
test_size = config["ml"]["test_size"]
random_state = config["ml"]["random_state"]
cv_folds = config["ml"]["cv_folds"]


students = load_processed_data("mat")
train_data, test_data = features.split_data(students, pass_mark, test_size, random_state)

print("Train:", len(train_data), "Test:", len(test_data))
print("Pass rate train/test:", train_data["passed"].mean().round(3), test_data["passed"].mean().round(3))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Train: 316 Test: 79
Pass rate train/test: 0.671 0.671


In [4]:
features_A  =  features.get_feature_columns(students,include_prior_grades=True)
features_B  =  features.get_feature_columns(students,include_prior_grades=False)
print(len(features_A), len(features_B))



32 30


In [5]:
def run_feature_set(feature_columns, name):

    X_train = features.prepare_inputs(train_data, feature_columns)
    X_test = features.prepare_inputs(test_data, feature_columns, list(X_train.columns))

    regression_models = train.train_models(
        train.get_regression_models(random_state), X_train, train_data["G3"])
    classification_models = train.train_models(
        train.get_classification_models(random_state), X_train, train_data["passed"])
    
    print("=====", name, "=====")
    print("Regression (predict G3). Lower MAE and RMSE is better:")
    display(evaluate.compare_models(regression_models, X_test, test_data["G3"], "regression"))
    print("Classification (predict pass). Higher is better:")
    display(evaluate.compare_models(classification_models, X_test, test_data["passed"], "classification"))

    return regression_models, classification_models, X_train

In [6]:
reg_A, clf_A ,X_train_A = run_feature_set(features_A, "SET A (with G1, G2)")

===== SET A (with G1, G2) =====
Regression (predict G3). Lower MAE and RMSE is better:


,mae,rmse,r2
Random Forest,1.026,1.571,0.877
Decision Tree,1.075,1.786,0.841
Linear Regression,1.515,2.130,0.774
Baseline (mean),3.322,4.482,-0.000


Classification (predict pass). Higher is better:


,accuracy,precision,recall,f1
Logistic Regression,0.886,0.958,0.868,0.911
Random Forest,0.873,0.957,0.849,0.900
Decision Tree,0.873,0.978,0.830,0.898
Baseline (always pass),0.671,0.671,1.000,0.803


In [7]:
reg_B, clf_B , X_train_B= run_feature_set(features_B, "B (without G1, G2)")

===== B (without G1, G2) =====
Regression (predict G3). Lower MAE and RMSE is better:


,mae,rmse,r2
Random Forest,2.979,3.891,0.246
Decision Tree,3.157,4.080,0.171
Baseline (mean),3.322,4.482,-0.000
Linear Regression,3.724,4.677,-0.089


Classification (predict pass). Higher is better:


,accuracy,precision,recall,f1
Baseline (always pass),0.671,0.671,1.000,0.803
Random Forest,0.671,0.690,0.925,0.790
Logistic Regression,0.658,0.717,0.811,0.761
Decision Tree,0.646,0.719,0.774,0.745


In [8]:
print("Regression: average RMSE over 5 splits (lower is better)")
display(evaluate.cross_validated_scores(
    train.get_regression_models(random_state),
    X_train_B, train_data["G3"], "regression", cv_folds, random_state))

print("Classification: average F1 over 5 splits (higher is better)")
display(evaluate.cross_validated_scores(
    train.get_classification_models(random_state),
    X_train_B, train_data["passed"], "classification", cv_folds, random_state))

Regression: average RMSE over 5 splits (lower is better)


,rmse_mean,rmse_std
Baseline (mean),4.578,0.448
Linear Regression,4.325,0.323
Decision Tree,4.619,0.504
Random Forest,3.879,0.390


Classification: average F1 over 5 splits (higher is better)


,f1_mean,f1_std
Baseline (always pass),0.803,0.004
Logistic Regression,0.776,0.020
Decision Tree,0.790,0.024
Random Forest,0.800,0.024


In [11]:
X_test_B = features.prepare_inputs(test_data, features_B, list(X_train_B.columns))
evaluate.compare_models(clf_B, X_test_B, test_data["passed"], "classification", pos_label=0)

,accuracy,precision,recall,f1
Decision Tree,0.646,0.455,0.385,0.417
Logistic Regression,0.658,0.474,0.346,0.400
Random Forest,0.671,0.500,0.154,0.235
Baseline (always pass),0.671,0.000,0.000,0.000


In [ ]:
# (which inputs does the Random Forest use most?)
print("Set A:")
print(train.feature_importances(reg_A["Random Forest"], list(X_train_A.columns)))
print()
print("Set B:")
print(train.feature_importances(reg_B["Random Forest"], list(X_train_B.columns)))

Set A:
G2                 0.794
absences           0.122
age                0.011
G1                 0.007
guardian_mother    0.006
Walc               0.006
reason_home        0.005
health             0.004
famrel             0.004
goout              0.003
dtype: float64

Set B:
absences    0.198
failures    0.186
goout       0.039
health      0.037
age         0.037
Walc        0.036
freetime    0.033
Medu        0.033
Fedu        0.021
famrel      0.021
dtype: float64


In [12]:
students_with_marks = train_data[train_data["G3"] > 0]
X_train_no_zero = features.prepare_inputs(
    students_with_marks, features_B, list(X_train_B.columns))

models_no_zero = train.train_models(
    train.get_regression_models(random_state),
    X_train_no_zero, students_with_marks["G3"])

evaluate.compare_models(models_no_zero, X_test_B, test_data["G3"], "regression")

,mae,rmse,r2
Random Forest,3.152,4.394,0.038
Decision Tree,3.333,4.541,-0.027
Baseline (mean),3.463,4.634,-0.069
Linear Regression,3.545,4.729,-0.114
